# Day 11

We will see the difference in time between numpy and python list

In [5]:
import numpy as np
import math
import random 

In [6]:
# Python list

In [31]:
class MLP:
    
    def __init__(self, layer_list, activation_list=None):
        self.params = [[[random.uniform(-1, 1)/(math.sqrt(layer_list[i-1])) for _ in range(layer_list[i])] for _ in range(layer_list[i-1])]for i in range(1,len(layer_list))]
        # adding for bias
        for i in range(1,len(layer_list)):
            self.params[i-1].append([0]*layer_list[i])
        self.param_grad = [[[0 for _ in range(layer_list[i])] for _ in range(layer_list[i-1]+1)]for i in range(1,len(layer_list))]
        self.activation_list = activation_list
        self.Q = [[0] * layer_list[i] for i in range(1,len(layer_list))]
        self.input_value = None
        self.output_values = None
        self.layer_list = layer_list
        
    def forward(self, inputs):
        self.input_values = inputs
        # Reset if already has some value
        self.Q = [[0] * self.layer_list[i] for i in range(1,len(self.layer_list))]
        # layer output will have output length as no. of neurons i.e. self.neurons
        self.output_values = [[] for _ in range(len(self.layer_list))]
        self.output_values[0] = inputs + [1.0]
        for k in range(len(self.layer_list)-1):
            # we have 2d array of parameters and 1 d array of inputs
            # parameters -> (n_inputs+1, n_neurons), inputs -> (n_inputs+1)
            for j in range(self.layer_list[k+1]):
                for i in range(self.layer_list[k]+1):
                    self.Q[k][j] += self.params[k][i][j] * self.output_values[k][i]
            if self.activation_list[k] == 'tanh':
                self.output_values[k+1] =  [math.tanh(Qi) for Qi in self.Q[k]] 
            elif self.activation_list[k] == 'ReLU':
                self.output_values[k+1] = [max(Qi, 0) for Qi in self.Q[k]]
            else:
                self.output_values[k+1] = self.Q[k]
            self.output_values[k+1] += [1.0]
        return self.output_values[-1][:-1]
        

    def backward(self, loss_grad):
        
        # dfdq = [[1] * layer_list[i] for i in range(1,len(layer_list))]
        grad_in = loss_grad
        for k in range(len(self.layer_list)-1,0,-1):
            dfdq = [1] * self.layer_list[k]
            if self.activation_list[k-1] == 'ReLU':
                dfdq = [1 if Q_i > 0 else 0 for Q_i in self.Q[k-1]]
            elif self.activation_list[k-1] == 'tanh':
                dfdq = [1 - math.tanh(Q_i)**2 for Q_i in self.Q[k-1]]
            dLdQ = [grad_i * dfdq_i for grad_i, dfdq_i in zip(grad_in, dfdq)]
            for j in range(self.layer_list[k]):
                for i in range(self.layer_list[k-1]+1):
                    self.param_grad[k-1][i][j] += dLdQ[j] * self.output_values[k-1][i]
            dLdX = [0] * (self.layer_list[k-1])
            for j in range(self.layer_list[k]):
                for i in range(self.layer_list[k-1]):
                    dLdX[i] += dLdQ[j] * self.params[k-1][i][j]
            grad_in = dLdX

    def parameters(self):
        return self.params

    def grads(self):
        return self.param_grad


class MSE:
    def __init__(self):
        pass

    def loss(self, y_pred, y_true):
        inputs = len(y_pred)
        ans = 0.0
        grads = []
        for y_predi, y_truei in zip(y_pred,y_true):
            ans += (y_predi-y_truei)**2 
            grads.append(2*(y_predi-y_truei))
        ans /= inputs
        for i in range(inputs):
            grads[i]/=inputs
        return (ans, grads)


class SGD:
    def __init__(self, parameters, grads, learning_rate=0.01):
        self.parameters = parameters
        self.grads = grads
        self.learning_rate = learning_rate

    def step(self):
        for layer_parameters, layer_grads in zip(self.parameters, self.grads):
            for neuron_parameters, neuron_grad in zip(layer_parameters, layer_grads):
                for i in range(len(neuron_parameters)):
                    neuron_parameters[i] -= self.learning_rate * neuron_grad[i]
                    
        
    def zero_grad(self):
        for layer_grads in self.grads:
            for neuron_grads in layer_grads:
                 for i in range(len(neuron_grads)):
                    neuron_grads[i] = 0



In [32]:
X = np.random.randn(1000, 100)  # 1000 samples, 100 features
y = np.random.randn(1000, 1)    # 1000 targets

In [33]:
X_list = X.tolist()
y_list = y.tolist()

In [34]:
import time
start = time.time()

nn = MLP([100, 256, 256, 1], ['tanh', 'tanh', 'tanh'])
loss_func = MSE()
optimizer = SGD(nn.parameters(), nn.grads(), 0.5)
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    for input_value, target in zip(X_list, y_list):
        output = nn.forward(input_value)
        loss, loss_grad = loss_func.loss(output, target)
        total_loss += loss
        nn.backward([xi/4 for xi in loss_grad])
    optimizer.step()
    optimizer.zero_grad()

end = time.time()
print(f"Time taken: {end - start:.2f} seconds")

Time taken: 131.65 seconds


In [35]:
# numpy
rg = np.random.default_rng()

In [36]:
class MLP_numpy:
    def __init__(self, layer_list, activation_list=None):
        self.w = [rg.uniform(-1,1,(layer_list[i], layer_list[i-1]))for i in range(1,len(layer_list))]
        self.w_grad = [np.zeros((layer_list[i], layer_list[i-1])) for i in range(1,len(layer_list))]
        self.b = [np.zeros(layer_list[i]).reshape(-1,1) for i in range(1,len(layer_list))]
        self.b_grad = [np.zeros(layer_list[i]).reshape(-1,1) for i in range(1,len(layer_list))]
        self.activation_list = activation_list
        self.Q = [np.zeros(layer_list[i]).reshape(1,-1) for i in range(0,len(layer_list))]
        self.output_values = None
        self.layer_list = layer_list
        
    def forward(self, inputs):
        # Reset if already has some value
        for i in range(len(self.layer_list)-1):
            self.Q[i][:] = 0
        # layer output will have output length as no. of neurons i.e. self.neurons
        self.output_values = [None] * len(self.layer_list)
        self.output_values[0] = np.array(inputs).reshape(-1,1)
        for k in range(1,len(self.layer_list)):
            self.Q[k] = self.w[k-1] @ self.output_values[k-1] + self.b[k-1]
            if self.activation_list[k-1] == 'tanh':
                self.output_values[k] = np.tanh(self.Q[k])
            elif self.activation_list[k-1] == 'ReLU':
                self.output_values[k] = np.clip(self.Q[k],a_min=0)
        
        return self.output_values[-1]
        
    def backward(self, loss_grad): 
        grad_in = loss_grad
        for l in range(len(self.layer_list)-1,0,-1):
            dfdq = np.ones_like(grad_in)
            if self.activation_list[l-1] == 'tanh':
                dfdq = (1-((self.output_values[l])**2))
            elif self.activation_list[l-1] == 'ReLU':
                dfdq = (self.Q[l] > 0).astype(float)
            dLdQ = grad_in * dfdq
            self.b_grad[l-1] += dLdQ
            self.w_grad[l-1] += dLdQ @ self.output_values[l-1].T
            grad_in =  self.w[l-1].T @ dLdQ
            
            
    def parameters(self):
        return (self.w, self.b)
        
    def grads(self):
        return (self.w_grad, self.b_grad)

class MSE:
    def __init__(self):
        pass

    def loss(self, y_pred, y_true):
        inputs = len(y_pred)
        grads = (2*(y_pred - y_true))/inputs
        ans_matrix = (y_pred-y_true)**2
        ans = sum(ans_matrix)/inputs
        return (ans, grads)


class SGD:
    def __init__(self, weights, bias, weights_grad, bias_grad, learning_rate=0.01):
        self.weights = weights
        self.bias = bias
        self.weights_grad = weights_grad
        self.bias_grad = bias_grad
        self.learning_rate = learning_rate

    def step(self):
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * self.weights_grad[i]
            self.bias[i] -= self.learning_rate * self.bias_grad[i]
                    
        
    def zero_grad(self):
        for i in range(len(self.weights_grad)):
            self.weights_grad[i][:] = 0
            self.bias_grad[i][:] = 0

In [37]:
import time
start = time.time()

nn = MLP_numpy([100, 256, 256, 1], ['tanh', 'tanh', 'tanh'])
loss_func = MSE()
optimizer = SGD(nn.parameters()[0],nn.parameters()[1] , nn.grads()[0], nn.grads()[1], 0.5)
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    for input_value, target in zip(X, y):
        output = nn.forward(input_value)
        # print(f"output {output}")
        loss, loss_grad = loss_func.loss(output, target)
        total_loss += loss
        nn.backward(loss_grad)
    optimizer.step()
    optimizer.zero_grad()

end = time.time()
print(f"Time taken: {end - start:.2f} seconds")

Time taken: 1.35 seconds
